In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [ ]:
db_path='Data/inventory2.db'
conn=sqlite3.connect(r'C:\Users\sk021\Desktop\Project 2\Data\inventory2.db')

In [ ]:
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print(f"Table Name : {table}")
    df=pd.read_sql_query(f"select * from {table} limit 5",conn)
    display(df)

In [ ]:
df = pd.read_sql_query("""
SELECT
    VendorNumber,
    VendorName,
    PONumber,
    PODate,
    ReceivingDate,
    InvoiceDate,
    PayDate,
    PurchasePrice,
    Quantity,
    Dollars
FROM purchases
""", conn)

print(df)

In [ ]:
purchases_aggregated_df=pd.read_sql_query('''
select p.PONumber,
count (distinct p.Brand ) as total_brands,
sum (p.quantity) as total_quantity,
sum (p.dollars) as total_item_dollars,
avg (julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_reciveing_delay
from purchases p
group by p.PONumber
''',conn)

In [ ]:
purchases_aggregated_df

In [ ]:
pd.read_sql_query("""
select 
vi.PONumber,
vi.Quantity,
vi.Dollars,
vi.Freight,
(julianday(vi.InvoiceDate) - julianday(vi.PODate)) as day_po_invoice,
(julianday(vi.PayDate) -  julianday(vi.InvoiceDate)) as day_to_pay
from vendor_invoice vi
""",conn)

In [ ]:
df=pd.read_sql_query(""" 
with purchasessss_aggregated_df as (
select p.PONumber,
count (distinct p.Brand ) as total_brands,
sum (p.quantity) as total_quantity,
sum (p.dollars) as total_item_dollars,
avg (julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_reciveing_delay
from purchases p
group by p.PONumber
)
select 
vi.PONumber,vi.Quantity as invoice_quantity,vi.Dollars as invoice_dollar,vi.Freight,
(julianday(vi.InvoiceDate) - julianday(vi.PODate)) as day_po_invoice,(julianday(vi.PayDate) -  julianday(vi.InvoiceDate)) as day_to_pay,
pa.total_brands,pa.total_quantity,pa.total_item_dollars,pa.avg_reciveing_delay
from vendor_invoice vi
left join 
purchasessss_aggregated_df pa
on
vi.PONumber=pa.PONumber

""",conn)

In [ ]:
display(df.head)

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
def create_invoice_risk_lable(row):
    # invoice total mismath with item-level total
    if (abs(row['invoice_dollar'] - row['total_item_dollars']) > 5):
        return 1
    # abnormally high recieving delay
    if row['avg_reciveing_delay'] > 10:
        return 1
    return 0
df['flag_invoice']=df.apply(create_invoice_risk_lable,axis=1)
df['flag_invoice'].value_counts()

In [ ]:
df['flag_invoice'].value_counts().plot(kind='bar')

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(df.iloc[:,1:-1].corr(),annot=True)
plt.show()

In [ ]:
flaged=df[df['flag_invoice']==1]
normal=df[df['flag_invoice']==0]

In [ ]:
significant_features = []
non_significant_features = []
result=[]

In [ ]:
df.columns

In [ ]:
metrics=['invoice_quantity','invoice_dollar','Freight','day_po_invoice',
         'day_to_pay','total_brands','total_quantity','total_item_dollars','avg_reciveing_delay','flag_invoice']

In [ ]:
from scipy.stats import ttest_ind
for metric in metrics:
    flaged_mean=flaged[metric].mean()
    normal_mean=normal[metric].mean()
    t_stat,p_value=ttest_ind(
        flaged[metric].dropna(),
        normal[metric].dropna(),
        equal_var=False
    )
    if p_value < 0.05:
        significant_features.append(metric)
        result.append({
            "metric": metric,
            "falged_mean" : flaged_mean.round(2),
            "Normal_mean" : normal_mean.round(2),
            "P_value" : p_value
        })
    else:
        non_significant_features.append(metric)
        print(f"metric: {metric}")
        print({
            'metric' : metric,
            "flaged_mean" : flaged_mean,
            "Normal_mean" : normal_mean,
            "p_value" : p_value,
            "t-stat" : t_stat
        })
        
    

In [ ]:
result

In [ ]:
significant_features

In [ ]:
non_significant_features

In [ ]:
df.columns

In [ ]:
X=df[['invoice_quantity','invoice_dollar','Freight','total_brands','total_quantity','day_po_invoice','total_item_dollars']]
y=df['flag_invoice']

In [ ]:
X.describe().round()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X,y,random_state=42)

In [ ]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model1=LogisticRegression()
model1.fit(x_train,y_train)

model2=DecisionTreeClassifier()
model2.fit(x_train,y_train)

model3=RandomForestClassifier()
model3.fit(x_train,y_train)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

In [ ]:
def evaluate_model(model, X_test, y_test,name):
    print(name)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Acuraccy : {accuracy * 100 : .2f}%")
    print(classification_report(y_test, y_pred))

In [ ]:
evaluate_model(model1,x_test_scaled,y_test)

In [ ]:
evaluate_model(model2,x_test_scaled,y_test)

In [ ]:
evaluate_model(model3,x_test_scaled,y_test)

In [ ]:
model3.feature_importances_

In [ ]:
feature_importances=pd.DataFrame({
    "Feature" : x_train.columns,
    "importance" : model3.feature_importances_
}).sort_values(by='importance',ascending=False)

In [ ]:
feature_importances

In [ ]:
X=df[['invoice_quantity','invoice_dollar','Freight','total_quantity','total_item_dollars']]
y=df['flag_invoice']

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
model3=RandomForestClassifier(n_estimators=1000)
model3.fit(x_train,y_train)

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Acuraccy : {accuracy * 100 : .2f}%")
    print(classification_report(y_test, y_pred))

In [ ]:
evaluate_model(model3,x_test_scaled,y_test)

In [ ]:
param_grid={
    'n_estimators': [100,200,3000],
    'max_depth' : [None,4,5,6],
    'min_samples_split' : [2,3,5,],
    'min_samples_leaf':[1,2,5],
    'criterion': ['gini','entropy']
}

In [ ]:
from sklearn.metrics import make_scorer,f1_score

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
rf=RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

In [ ]:
scorer=make_scorer(f1_score)

In [ ]:
grid_search=GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    
)

In [ ]:
grid_search.fit(x_train_scaled,y_train)

In [ ]:
evaluate_model(x_test_scaled,y_test,'Random Forest Classifier')

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
cm=confusion_matrix(grid_search.predict,y_test)
plt.figure(figsize=(10,5))
sns.heatmap(cm,innot=True,cbar=Fale)
plt.show()

In [ ]:
grid_search.best_params_